## 2 — Gráficos: Visão Geral do Campus Restinga

**Seção 2.5 - Indicadores e matrículas:**



| Gráfico | Tipo | Descrição |
|---------|------|-----------|
| 1 | Linha | Evolução de TC, TE, TR, TPE e IEf por ano |
| 2 | Linha | Evolução de MREG e MRET por ano |
| 3 | Barras empilhadas | Matrículas atendidas por ano e categoria |
| 4 | Barras empilhadas | Matrículas por curso e ano |
| 5 | Linha | Ingressantes e Concluintes por ano |

**Seção 2.6 -  Evasão:**

| Gráfico | Tipo | Descrição |
|---------|------|-----------|
| 6 | Barras empilhadas | Motivos de saída por ano |
| 7 | Barras horizontais empilhadas % | Proporção dos motivos de evasão por curso (último ano) |

**Seção 2.7 -  Conclusão:**

| Gráfico | Tipo | Descrição |
|---------|------|-----------|
| 8  | Barras empilhadas | Conclusões no Prazo e Conclusões com Atraso por Curso |
| 9 | Barras horizontais | Tempo mediano de conclusão por curso |
| 10 | Boxplot | Distribuição do tempo até conclusão por curso |


#### 2.1. Importações, configurações e paletas

In [1]:
import pandas as pd
import plotly.express as px
import sys
import os

sys.path.append(os.path.abspath(".."))

from utils import (
    carregar_dados,
    calcular_indicadores,
    gerar_mapa_cores,
    aplicar_layout_light,
    CORES_INDICADORES,
    CORES_CATEGORIA,
    CORES_CONCLUSAO_LIGHT,
    CORES_FLUXO_LIGHT,
    CORES_MATRICULAS_ATIVAS_LIGHT,
    CORES_SITUACAO_LIGHT,
    COR_TEMPO_MEDIANO,
    CORES_CURSO,
    CORES_SITUACAO,
    PALETA_QUALITATIVA_LIGHT,
    ROTULOS_INDICADORES
)

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

2026-06-24 11:11:16.927 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


#### 2.2 Carga dos dados tratados


In [2]:
df_completo = pd.read_parquet("../dados_tratados/restinga_dados_tratados.parquet")
print("Shape:", df_completo.shape)
df_completo.head(3)

Shape: (12022, 18)


,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno
0,2017,Em curso,Não declarada,44681986,1207788,2014-12-22,2012-03-05,2012-03-01,Informação e Comunicação,35 a 39 anos,2018-03-05,Análise e Desenvolvimento de Sistemas,Não declarada,F,Retido,Informática,Superior-Tecnologia,Matutino
1,2017,Em curso,Branca,66262145,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino
2,2017,Em curso,Branca,66262155,2007734,2019-12-20,2016-02-22,2016-02-01,Controle e Processos Industriais,15 a 19 anos,2016-02-01,Técnico em Eletrônica,"0,5<RFP<=1",F,Em fluxo,Elétrica,Técnico-Integrado,Matutino


#### 2.3. Filtros utilizados

In [3]:
ANOS_DISPONIVEIS = sorted(df_completo["Ano"].dropna().unique())
TIPOS_DISPONIVEIS = sorted(df_completo["Tipo de Curso"].dropna().unique())
CURSOS_DISPONIVEIS = sorted(df_completo["Nome de Curso"].dropna().unique())

# Valores padrão: todos os anos, tipos e cursos.
PERIODO_ANALISE = (int(min(ANOS_DISPONIVEIS)), int(max(ANOS_DISPONIVEIS)))
TIPOS_SELECIONADOS = TIPOS_DISPONIVEIS
CURSOS_SELECIONADOS = CURSOS_DISPONIVEIS

# Para testar um filtro específico, descomentar e editar abaixo, por exemplo:
# PERIODO_ANALISE = (2020, 2024)
# TIPOS_SELECIONADOS = ["Tecnologia"]
# CURSOS_SELECIONADOS = ["Eletrônica Industrial"]

df = df_completo[
    (df_completo["Ano"] >= PERIODO_ANALISE[0])
    & (df_completo["Ano"] <= PERIODO_ANALISE[1])
    & (df_completo["Tipo de Curso"].isin(TIPOS_SELECIONADOS))
    & (df_completo["Nome de Curso"].isin(CURSOS_SELECIONADOS))
].copy()

CORES_CURSO_LIGHT = gerar_mapa_cores(df["Nome de Curso"])

print(f"Período: {PERIODO_ANALISE[0]} a {PERIODO_ANALISE[1]}")
print(f"Tipos selecionados: {len(TIPOS_SELECIONADOS)}")
print(f"Cursos selecionados: {len(CURSOS_SELECIONADOS)}")
print("Registros filtrados:", len(df))

Período: 2017 a 2025
Tipos selecionados: 5
Cursos selecionados: 11
Registros filtrados: 12022


In [4]:
df_completo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12022 entries, 0 to 12021
Data columns (total 18 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Ano                              12022 non-null  int64         
 1   Categoria da Situação            12022 non-null  object        
 2   Cor / Raça                       12022 non-null  object        
 3   Código da Matricula              12022 non-null  int64         
 4   Código do Ciclo Matricula        12022 non-null  int64         
 5   Data de Fim Previsto do Ciclo    12022 non-null  datetime64[ns]
 6   Data de Inicio do Ciclo          12022 non-null  datetime64[ns]
 7   Data de Ocorrencia da Matricula  12022 non-null  datetime64[ns]
 8   Eixo Tecnológico                 12022 non-null  object        
 9   Faixa Etária                     12022 non-null  object        
 10  Mês De Ocorrência da Situação    12022 non-null  datetime6

#### 2.4. Cálculo dos indicadores
Calcula os indicadores de Permanência e Êxito por diferentes agrupamentos.

In [5]:
ind_ano = calcular_indicadores(df, ["Ano"])
ind_ano_curso = calcular_indicadores(df, ["Ano", "Nome de Curso"])

ind_ano[["Ano", "TC", "TE", "TR", "TPE", "IEf", "MREG", "MRET"]]

,Ano,TC,TE,TR,TPE,IEf,MREG,MRET
0,2017,8.38,15.78,5.18,78.55,34.34,569,42
1,2018,7.80,9.84,7.70,82.36,43.96,765,79
2,2019,6.07,11.05,5.68,83.19,35.45,991,73
3,2020,0.16,7.52,13.57,78.66,2.06,960,166
4,2021,2.50,5.70,23.89,70.41,30.43,762,268
5,2022,7.65,9.49,35.38,55.12,44.62,695,518
6,2023,10.13,5.34,33.13,61.53,65.46,827,533
7,2024,5.56,2.35,35.36,62.18,70.29,988,617
8,2025,6.51,10.13,23.83,66.03,39.10,1034,414


#### 2.5 — Gráficos dos Indicadores e Matrículas

#### Gráfico 1 — Evolução dos Indicadores por Ano

Mostra a trajetória temporal dos principais indicadores acadêmicos de Permanência e Êxito. Permite visualizar simultaneamente permanência, conclusão, evasão, retenção e eficiência.

In [6]:
# melt() transforma o DataFrame de formato largo (uma coluna por indicador) para formato longo
ind_longo = ind_ano.melt(
    id_vars="Ano",
    value_vars=["TC", "TE", "TR", "TPE", "IEf"],
    var_name="Indicador",
    value_name="Valor (%)",
)

fig_g1 = px.line(
    ind_longo,
    x="Ano",
    y="Valor (%)",
    color="Indicador",
    markers=True,
    color_discrete_map=CORES_INDICADORES,
    title="1 — Evolução dos Indicadores por Ano",
    labels={"Indicador": "", "Valor (%)": "(%)"},
)
for trace in fig_g1.data:
    trace.name = ROTULOS_INDICADORES.get(trace.name, trace.name)
    trace.hovertemplate = trace.hovertemplate.replace(
        trace.legendgroup,
        ROTULOS_INDICADORES.get(trace.legendgroup, trace.legendgroup),
    )

fig_g1.update_xaxes(tickmode="linear", dtick=1)
fig_g1.update_layout(hovermode="x unified")
aplicar_layout_light(fig_g1, legenda_y=-0.2)
fig_g1.show()

ind_longo.head()

,Ano,Indicador,Valor (%)
0,2017,TC,8.38
1,2018,TC,7.80
2,2019,TC,6.07
3,2020,TC,0.16
4,2021,TC,2.50


#### Gráfico 2 — Matrículas Ativas Regulares (MREG) e Retidas (MRET) por Ano

Esse gráfico separa estudantes ainda em fluxo regular daqueles que já ultrapassaram o prazo previsto, apoiando a leitura da retenção como fenômeno de acompanhamento contínuo.

In [7]:
matr_ativas_longo = ind_ano.melt(
    id_vars="Ano",
    value_vars=["MREG", "MRET"],
    var_name="Tipo",
    value_name="Matrículas",
)

fig_g2 = px.line(
    matr_ativas_longo,
    x="Ano",
    y="Matrículas",
    color="Tipo",
    markers=True,
    color_discrete_map=CORES_MATRICULAS_ATIVAS_LIGHT,
    title="2 — Matrículas Ativas Regulares (MREG) e Retidas (MRET) por Ano",
    labels={"Tipo": ""},
)
fig_g2.update_xaxes(tickmode="linear", dtick=1)
fig_g2.update_layout(hovermode="x unified")
aplicar_layout_light(fig_g2, legenda_y=-0.2)
fig_g2.show()

matr_ativas_longo.head()

,Ano,Tipo,Matrículas
0,2017,MREG,569
1,2018,MREG,765
2,2019,MREG,991
3,2020,MREG,960
4,2021,MREG,762


#### Gráfico 3 — Matrículas por Ano e Categoria

Mostra o volume de matrículas distribuído entre estudantes em curso, concluintes e evadidos.

In [8]:
mat_cat = (
    df.groupby(["Ano", "Categoria da Situação"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

fig_g3 = px.bar(
    mat_cat,
    x="Ano",
    y="Qtd",
    color="Categoria da Situação",
    barmode="stack",
    color_discrete_map=CORES_CATEGORIA,
    title="3 — Matrículas por Ano e Categoria",
    labels={"Qtd": "Matrículas", "Categoria da Situação": ""},
    text_auto=True,
)
fig_g3.update_xaxes(tickmode="linear", dtick=1)
fig_g3.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5))
aplicar_layout_light(fig_g3, legenda_y=-0.2)
fig_g3.show()

mat_cat.head()

,Ano,Categoria da Situação,Qtd
0,2017,Concluintes,70
1,2017,Em curso,611
2,2017,Evadidos,130
3,2018,Concluintes,80
4,2018,Em curso,844


##### Gráfico 4 — Ingressantes e Concluintes por Ano



In [9]:
fluxo_longo = ind_ano.melt(
    id_vars="Ano",
    value_vars=["ingressantes", "concluintes"],
    var_name="Tipo",
    value_name="Quantidade",
)
fluxo_longo["Tipo"] = fluxo_longo["Tipo"].map({
    "ingressantes": "Ingressantes",
    "concluintes": "Concluintes",
})

fig_g4 = px.line(
    fluxo_longo,
    x="Ano",
    y="Quantidade",
    color="Tipo",
    markers=True,
    color_discrete_map=CORES_FLUXO_LIGHT,
    title="4 — Ingressantes e Concluintes por Ano",
    labels={"Tipo": "", "Quantidade": "Matrículas"},
)
fig_g4.update_xaxes(tickmode="linear", dtick=1)
fig_g4.update_layout(hovermode="x unified")
aplicar_layout_light(fig_g4, legenda_y=-0.2)
fig_g4.show()

fluxo_longo.head()

,Ano,Tipo,Quantidade
0,2017,Ingressantes,292
1,2018,Ingressantes,344
2,2019,Ingressantes,373
3,2020,Ingressantes,218
4,2021,Ingressantes,147


##### Gráfico 5 — Matrículas Atendidas por Curso e Ano

In [10]:
mat_curso_ano = (
    df.groupby(["Ano", "Nome de Curso"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

fig_g5 = px.bar(
    mat_curso_ano,
    x="Ano",
    y="Qtd",
    color="Nome de Curso",
    color_discrete_map=CORES_CURSO_LIGHT,
    barmode="stack",
    title="5 — Matrículas Atendidas por Curso e Ano",
    labels={"Qtd": "Matrículas", "Nome de Curso": "Curso"},
    #text_auto=True,
)
fig_g5.update_xaxes(tickmode="linear", dtick=1)
aplicar_layout_light(fig_g5, legenda_y=-0.25)
fig_g5.show()

mat_curso_ano.head()


,Ano,Nome de Curso,Qtd
0,2017,Análise e Desenvolvimento de Sistemas,247
1,2017,Eletrônica Industrial,84
2,2017,Gestão Desportiva e de Lazer,112
3,2017,Letras - Português e Espanhol,30
4,2017,Técnico em Comércio,33


#### 2.6 Gráficos Evasão

#### Gráfico 6 — Motivos de Evasão por Ano

Exibe os motivos de evasão em valores absolutos.

In [16]:
situacoes_evasao = ["Abandono", "Desligada", "Transf. externa", "Transf. interna"]

df_evasoes = (
    df[df["Situação de Matrícula"].isin(situacoes_evasao)]
    .groupby(["Ano", "Situação de Matrícula"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

fig_g6 = px.bar(
    df_evasoes,
    x="Ano",
    y="Qtd",
    color="Situação de Matrícula",
    barmode="stack",
    color_discrete_map=CORES_SITUACAO_LIGHT,
    title="6 — Motivos de Saída por Ano",
    labels={"Qtd": "Evadidos", "Situação de Matrícula": "Motivo"},
    text_auto=True,
)
fig_g6.update_xaxes(tickmode="linear", dtick=1)
aplicar_layout_light(fig_g6, legenda_y=-0.3)
fig_g6.show()

df_evasoes.head()

,Ano,Situação de Matrícula,Qtd
0,2017,Abandono,70
1,2017,Desligada,45
2,2017,Transf. externa,13
3,2017,Transf. interna,2
4,2018,Abandono,40


#### Gráfico 7 — Motivos de Evasão por Curso no último ano

Volume absoluto de evasões por motivo e por curso no último ano disponível, ordenando os cursos pelo total de evasões.

In [17]:
ultimo_ano = int(PERIODO_ANALISE[1])
motivos_ev = ["Abandono", "Desligada", "Transf. externa"]

df_ev_ano = df_completo[
    (df_completo["Ano"] == ultimo_ano)
    & (df_completo["Tipo de Curso"].isin(TIPOS_SELECIONADOS))
    & (df_completo["Nome de Curso"].isin(CURSOS_SELECIONADOS))
    & (df_completo["Situação de Matrícula"].isin(motivos_ev))
]

ev_curso = (
    df_ev_ano
    .groupby(["Nome de Curso", "Situação de Matrícula"])["Código da Matricula"]
    .count()
    .reset_index(name="Qtd")
)

tot_ev = ev_curso.groupby("Nome de Curso")["Qtd"].sum().reset_index(name="Total")
ev_curso = ev_curso.merge(tot_ev, on="Nome de Curso")
ordem_cursos = tot_ev.sort_values("Total", ascending=False)["Nome de Curso"].tolist()

fig_g7 = px.bar(
    ev_curso,
    y="Nome de Curso",
    x="Qtd",
    color="Situação de Matrícula",
    orientation="h",
    barmode="stack",
    color_discrete_map=CORES_SITUACAO_LIGHT,
    text_auto=True,
    title=f"7 — Motivos de Evasão por Curso ({ultimo_ano})",
    labels={"Qtd": "Evadidos", "Nome de Curso": "", "Situação de Matrícula": "Motivo"},
    category_orders={"Nome de Curso": ordem_cursos},
)
aplicar_layout_light(fig_g7, legenda_y=-0.2)
fig_g7.show()

ev_curso.head()

,Nome de Curso,Situação de Matrícula,Qtd,Total
0,Análise e Desenvolvimento de Sistemas,Abandono,9,19
1,Análise e Desenvolvimento de Sistemas,Desligada,10,19
2,Eletrônica Industrial,Desligada,5,7
3,Eletrônica Industrial,Transf. externa,2,7
4,Gestão Desportiva e de Lazer,Abandono,45,46


#### 2.6 Gráficos Conclusão

#### Gráfico 8 — Conclusões no Prazo e Conclusões com Atraso por Curso

In [13]:
situacoes_conclusao = ["Concluída no prazo", "Concluída com atraso"]

df_conc = df[
    df["Situação de Matrícula"].isin(situacoes_conclusao)
].copy()

df_conc["Anos_Conclusao"] = (
    (df_conc["Mês De Ocorrência da Situação"] - df_conc["Data de Inicio do Ciclo"])
    .dt.days / 365.25
).round(2)

df_conc = df_conc[
    (df_conc["Anos_Conclusao"] > 0)
    & (df_conc["Anos_Conclusao"] < 15)
]

conc_prazo = (
    df_conc
    .value_counts(["Nome de Curso", "Situação de Matrícula"])
    .reset_index(name="Qtd")
)

ordem_conclusao = (
    conc_prazo
    .groupby("Nome de Curso")["Qtd"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

fig_g8 = px.bar(
    conc_prazo,
    x="Nome de Curso",
    y="Qtd",
    color="Situação de Matrícula",
    barmode="stack",
    color_discrete_map=CORES_CONCLUSAO_LIGHT,
    title="8 — Conclusões no Prazo e Conclusões com Atraso por Curso",
    labels={"Qtd": "Concluintes", "Nome de Curso": "", "Situação de Matrícula": "Situação"},
    text_auto=True,
    category_orders={"Nome de Curso": ordem_conclusao},
)

fig_g8.update_layout(xaxis_tickangle=-30)
aplicar_layout_light(fig_g8, legenda_y=-0.5)
fig_g8.show()

conc_prazo.head()

,Nome de Curso,Situação de Matrícula,Qtd
0,Técnico em Lazer,Concluída no prazo,125
1,Técnico em Guia de Turismo,Concluída no prazo,114
2,Técnico em Eletrônica,Concluída no prazo,71
3,Técnico em Informática,Concluída no prazo,70
4,Análise e Desenvolvimento de Sistemas,Concluída com atraso,68


#### Gráfico 9 — Tempo Mediano de Conclusão por Curso

In [14]:
if len(df_conc) > 0:
        tempo_mediano = (
            df_conc
            .groupby(["Nome de Curso", "Tipo de Curso"], as_index=False)
            .agg(
                Mediana=("Anos_Conclusao", "median"),
                Media=("Anos_Conclusao", "mean"),
                N=("Anos_Conclusao", "count")
            )
            .round(2)
            .sort_values("Mediana")
        )

        fig_g9 = px.bar(
            tempo_mediano,
            x="Mediana",
            y="Nome de Curso",
            orientation="h",
            text="Mediana",
            title="9 — Tempo Mediano de Conclusão por Curso",
            labels={"Mediana": "Tempo Mediano de Conclusão (anos)", "Nome de Curso": ""},
            custom_data=["Media", "N", "Tipo de Curso"],
        )

        fig_g9.update_traces(
            marker_color=COR_TEMPO_MEDIANO,
            texttemplate="%{x:.2f}",
            textposition="outside",
            hovertemplate=(
                "<b>%{y}</b><br>"
                "Mediana: %{x:.2f} anos<br>"
                "Média: %{customdata[0]:.2f} anos<br>"
                "Concluintes: %{customdata[1]}<br>"
                "Tipo: %{customdata[2]}<extra></extra>"
            ),
        )

        fig_g9.update_layout(
            showlegend=False,
            xaxis=dict(range=[0, tempo_mediano["Mediana"].max() * 1.2]),
        )

        aplicar_layout_light(fig_g9)
        fig_g9.show()

else:
    print("Sem dados de concluídos para o filtro selecionado.")


tempo_mediano.head()

,Nome de Curso,Tipo de Curso,Mediana,Media,N
8,Técnico em Guia de Turismo,Técnico-Subsequente,1.43,1.69,142
10,Técnico em Lazer,Técnico-Integrado,2.80,2.93,141
2,Gestão Desportiva e de Lazer,Superior-Tecnologia,3.75,4.17,43
7,Técnico em Eletrônica,Técnico-Integrado,3.84,4.17,95
9,Técnico em Informática,Técnico-Integrado,3.97,4.07,88


#### Gráfico 10 — Distribuição do Tempo até Conclusão por Curso

In [15]:
if len(df_conc) > 0:
        ordem_box = (
            df_conc
            .groupby("Nome de Curso")["Anos_Conclusao"]
            .median()
            .sort_values(ascending=False)
            .index
            .tolist()
        )

        fig_g10 = px.box(
            df_conc,
            x="Nome de Curso",
            y="Anos_Conclusao",
            color="Nome de Curso",
            color_discrete_map=CORES_CURSO_LIGHT,
            points="outliers",
            title="10 — Distribuição do Tempo até Conclusão por Curso",
            labels={"Anos_Conclusao": "Anos até Conclusão","Nome de Curso": ""},
            category_orders={"Nome de Curso": ordem_box},
        )

        fig_g10.update_layout(showlegend=False, xaxis_tickangle=-30)
        aplicar_layout_light(fig_g10)
        fig_g10.show()

else:
    print("Sem dados de concluídos para o filtro selecionado.")
    
df_conc.head()

,Ano,Categoria da Situação,Cor / Raça,Código da Matricula,Código do Ciclo Matricula,Data de Fim Previsto do Ciclo,Data de Inicio do Ciclo,Data de Ocorrencia da Matricula,Eixo Tecnológico,Faixa Etária,Mês De Ocorrência da Situação,Nome de Curso,Renda Familiar,Sexo,Situação de Matrícula,Subeixo Tecnológico,Tipo de Curso,Turno,Anos_Conclusao
31,2017,Concluintes,Branca,62534587,1957524,2017-07-31,2014-07-28,2014-07-01,"Turismo, Hospitalidade e Lazer",45 a 49 anos,2017-08-01,Gestão Desportiva e de Lazer,"0<RFP<=0,5",F,Concluída no prazo,Lazer,Superior-Tecnologia,Matutino,3.01
34,2017,Concluintes,Branca,62534569,1957524,2017-07-31,2014-07-28,2014-07-01,"Turismo, Hospitalidade e Lazer",25 a 29 anos,2017-08-01,Gestão Desportiva e de Lazer,"1,5<RFP<=2,5",F,Concluída no prazo,Lazer,Superior-Tecnologia,Matutino,3.01
39,2017,Concluintes,Preta,62534583,1957524,2017-07-31,2014-07-28,2014-07-01,"Turismo, Hospitalidade e Lazer",50 a 54 anos,2017-08-01,Gestão Desportiva e de Lazer,"0<RFP<=0,5",F,Concluída no prazo,Lazer,Superior-Tecnologia,Matutino,3.01
40,2017,Concluintes,Branca,54918129,1722403,2016-03-18,2013-03-18,2013-03-01,Informação e Comunicação,20 a 24 anos,2017-08-01,Análise e Desenvolvimento de Sistemas,"0,5<RFP<=1",F,Concluída com atraso,Informática,Superior-Tecnologia,Matutino,4.37
41,2017,Concluintes,Branca,54918121,1722403,2016-03-18,2013-03-18,2013-03-01,Informação e Comunicação,20 a 24 anos,2017-12-15,Análise e Desenvolvimento de Sistemas,"1,5<RFP<=2,5",F,Concluída com atraso,Informática,Superior-Tecnologia,Matutino,4.74
